# Parametrización

<div style="text-align: justify;">

Dentro de este _notebook_ se desarrolla el paso de parametrización de las variantes de las metaheurísticas construidas durante este trabajo de titulación. Particularmente, se requiere obtener un conjunto de hiperparámetros para cada versión del _NSGA-II_ que se evaluará, con un enfoque que busca ser **lo más justo posible entre las variantes**, de modo de asegurar la mejor configuración para cada una y que su evaluación posterior sea consistente.

Bajo este enfoque, se reconocen los siguientes parámetros dentro de la implementación actual (Tabla X):

| Parámetro                | Descripción                                                            | Variante     | Calibración  | Rango |
|--------------------------|------------------------------------------------------------------------|--------------|--------------|-----------------------------------------------|
| `ge_matrix`              | Matriz de distancias de expresión ($1 - S_{GE}$)                       | Todas        | No requerida | -                                             |
| `bi_matrix`              | Matriz de distancias biológicas ($1 - S_{BI}$)                         | Todas        | No requerida | -                                             |
| `n`                      | Número de genes (orden de las matrices)                                | Todas        | No requerida | [~2.000, ~11.000] genes                       |
| `k`                      | Número de medoides (_clusters_) por solución                           | Todas        | No requerida | 4                                             |
| `pop_size`               | Tamaño de la población                                                 | Todas        | Requerida    | 50-250 individuos (Saltos de 10)              |
| `max_obj_function_calls` | Presupuesto global de evaluaciones, que es también el criterio de paro | Todas        | No requerida | 150.000 llamadas                              |
| `crossover_prob`         | Probabilidad de _crossover_ entre miembros de la población             | Todas        | Requerida    | [0,80; 0,90] (paso de 0,01)                   |
| `mutation_prob`          | Probabilidad de que un hijo presente mutación                          | Todas        | Requerida    | [0,01; 0,05] (paso de 0,01)                   |
| `seed`                   | Semilla del generador pseudoaleatorio                                  | Todas        | No requerida | 50 semillas al azar (derivadas de la fecha del día, AAAAMMDD), compartidas por todas las variantes |
| `pls_neighborhood`       | Fracción de $n$ que define el tamaño máximo del vecindario de PLS      | `nsga2_PLS`  | Requerida    | [0,05; 0,25] (paso de 0,01)                   |
| `pls_local_budget`       | Evaluaciones que PLS puede usar por generación                         | `nsga2_PLS`  | Requerida    | [5, 10] % del total                           |
| `mols_local_budget`      | Evaluaciones que MOLS puede usar por generación                        | `nsga2_MOLS` | Requerida    | [5, 10] % del total                           |
| `mode` (MOLS)            | Criterio de aceptación de vecinos: _large_ (L-MOLS) acepta si la solución no lo domina; _narrow_ (N-MOLS) acepta solo si el vecino la domina | `nsga2_MOLS` | Requerida    | `"large"`, `"narrow"`                         |
| `M_v`                    | Matriz sobre la que MOLS define el vecindario $M_V$: $D_{GE}$ (expresión), $D_{BI}$ (biológica) o $\sqrt{D_{GE}^2 + D_{BI}^2}$ (combinación de ambas) | `nsga2_MOLS` | Requerida    | `"GE"`, `"BI"`, `"BOTH"`                      |
| `neighborhood`           | Radio sobre la matriz $M_V$ seleccionada (`M_v`)                       | `nsga2_MOLS` | Requerida    | Percentil [10, 35] de la distribución de $M_V$ (paso de 1) |
| `gens_estancamiento`     | Generaciones consecutivas sin mejora que activan el modo de estancamiento | `nsga2_PNS`, `nsga2_DBS` | No requerida | 1 (constante)                      |
| `eps`                    | $\varepsilon$ de la matriz de discrepancia logarítmica                  | `nsga2_DBS`  | No requerida | $10^{-3}$ (constante)                         |
| `nF`                     | Número de frentes de Pareto que PNS conserva en el vecindario de cada medoide | `nsga2_PNS`  | Requerida    | [1, 5] frentes (paso de 1)                    |
| `mode`                   | Matriz de discrepancia $D_{mode}$ con la que DBS parte los genes en celdas (fijo durante la corrida) | `nsga2_DBS`  | Requerida    | `"bio"`, `"no_discrepancy"`, `"expresion"`    |
| `p`                      | Percentil de la celda del medoide, ordenada de menos a más discrepante con él, que define el vecindario en DBS | `nsga2_DBS`  | Requerida    | Percentil [5, 25] (paso de 1)                 |

Respecto a MOLS, ambos criterios de aceptación (**L-MOLS**, _large_, y **N-MOLS**, _narrow_) y las tres definiciones de $M_V$ se calibran dentro de un único estudio (`nsga2_MOLS`), tratando `mode` y `M_v` como parámetros categóricos. El radio `neighborhood` se calcula como percentil de la distribución de la matriz $M_V$ seleccionada, por lo que su escala depende de esta elección.

Respecto a DBS, el vecindario de un medoide es el percentil `p` de su celda con menor $|D_{mode}|$ (genes menos discrepantes con él) y solo se aceptan candidatas que mejoran ambos objetivos de la solución explorada. Si el hipervolumen de $F_0$ no mejora en una generación (estancamiento), el criterio se relaja a mejorar al menos un objetivo hasta que el hipervolumen vuelva a mejorar. El umbral de estancamiento (`gens_estancamiento`, también usado por PNS) y `eps` se mantienen constantes (1 y $10^{-3}$, respectivamente): `mode` y `p` son los únicos hiperparámetros de DBS que se calibran.

Cabe considerar que todos aquellos parámetros que requieren calibración serán parametrizados según el mejor rendimiento, medido como el hipervolumen final del frente $F_0$ (respecto al punto de referencia `HV_REFERENCE_POINT`, promediado entre los datasets de parametrización). Este es, además, el criterio para seleccionar los parámetros definitivos en la etapa de ejecución de los experimentos.

</div>

In [1]:
# 1. Clonar el repo
#!git clone https://github.com/BenjaminGonzalezH/Thesis.git /content/Thesis

# 2. Montar Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 3. Copiar las matrices desde Drive al directorio que usa el notebook
import shutil
from pathlib import Path

DRIVE_MATRICES = Path("/content/drive/MyDrive/")                   # <-- ajusta a tu carpeta en Drive
DEST_MATRICES  = Path("/content/Thesis/Datasets/Matrices")        # minúscula: así lo espera el notebook en Colab

DEST_MATRICES.mkdir(parents=True, exist_ok=True)

archivos = list(DRIVE_MATRICES.glob("*.npz"))
for f in archivos:
    shutil.copy2(f, DEST_MATRICES / f.name)

print(f"{len(archivos)} archivos copiados a {DEST_MATRICES}")
print(sorted(p.name for p in DEST_MATRICES.iterdir()))

0 archivos copiados a \content\Thesis\Datasets\Matrices
['A1_S_DB.npz', 'A1_S_DE.npz', 'A2_S_DB.npz', 'A2_S_DB_M.npz', 'A2_S_DE.npz', 'B1_S_DB.npz', 'B1_S_DB_M.npz', 'B1_S_DE.npz', 'B2_S_DB.npz', 'B2_S_DB_M.npz', 'B2_S_DE.npz', 'B3_S_DB.npz', 'clustered_heatmap_BI_A1.png', 'clustered_heatmap_BI_A1_M.png', 'clustered_heatmap_BI_A2.png', 'clustered_heatmap_BI_A2_M.png', 'clustered_heatmap_BI_B1.png', 'clustered_heatmap_BI_B1_M.png', 'clustered_heatmap_BI_B2.png', 'clustered_heatmap_BI_B2_M.png', 'clustered_heatmap_BI_B3.png', 'clustered_heatmap_BI_B3_M.png', 'clustered_heatmap_GE_A1.png', 'clustered_heatmap_GE_A2.png', 'clustered_heatmap_GE_B1.png', 'clustered_heatmap_GE_B2.png', 'clustered_heatmap_GE_B3.png']


In [2]:
######################################################
# MODULO DE PREPARACIÓN DEL AMBIENTE
# Detección de entorno (Colab / Local), rutas del repositorio, instalación
# de Optuna y configuración común del optimizador para todas las variantes.
#######################################################

############################
# Importaciones
############################
import json                                                 # Guardado de las semillas de parametrización.
import sys                                                  # Ajuste de sys.path para importar los módulos de Source Code.
import os                                                   # Llamadas al SO.
import subprocess                                           # Instalación de dependencias faltantes.
from datetime import date                                  # Semilla basada en la fecha del día.
import numpy  as np                                         # Aplicación de operaciones matematicas eficientes.
from pathlib import Path                                    # Aplicar un formato generalizado de direcciones en el script.

############################
# Detección de entorno (Colab / Local)
############################
# En Colab (incluido VSCode conectado a un runtime de Colab) el repositorio se
# encuentra clonado en /content/Thesis. En local, se busca la raíz del
# repositorio subiendo desde el directorio de trabajo actual.
try:
    import google.colab                                     # Módulo disponible solo dentro de runtimes de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError(f"No se encontró la raíz del repositorio (Datasets/ y Source Code/) desde {_cwd}")

# Administración de directorios de matrices y código fuente.
MATRICES_DIR = THESIS_DIR / "Datasets" / "Matrices"
print(f"[entorno] {'Colab' if IN_COLAB else 'Local'} → {THESIS_DIR}")

SOURCE_CODE_DIR = THESIS_DIR / "Source Code"
if str(SOURCE_CODE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_CODE_DIR))

############################
# Optuna (instalación si no está disponible)
############################
# plotly se requiere solo para optuna.visualization (historial de optimización,
# importancia de parámetros, etc.).
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna", "plotly"])
    import optuna

from optuna.samplers import TPESampler                      # Muestreador bayesiano (Tree-structured Parzen Estimator).
optuna.logging.set_verbosity(optuna.logging.WARNING)        # Se silencia el log por trial; el progreso se reporta desde el propio notebook.
print(f"[optuna] versión {optuna.__version__}")

############################
# Configuración común del optimizador
############################
SEED = int(date.today().strftime("%Y%m%d"))                # Semilla del día (AAAAMMDD): sampler de Optuna (TPE) y generación de SEEDS_PARAM.
N_SEEDS_PARAM = 50                                          # Cantidad de semillas de las ejecuciones del NSGA-II.
SEEDS_PARAM = [int(s) for s in np.random.default_rng(SEED).integers(0, 2**31 - 1, size=N_SEEDS_PARAM)]
                                                            # 50 semillas al azar (derivadas de SEED), COMPARTIDAS por todas las variantes y trials.

# Almacenamiento persistente (SQLite): permite reanudar un estudio si el
# runtime se corta. En Colab el disco es efímero, por lo que conviene que
# RESULTS_DIR apunte a Google Drive si se quiere conservar entre sesiones.
RESULTS_DIR = THESIS_DIR / "Results" / "Parametrizacion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Respaldo en Google Drive (solo en Colab): semillas, estudios (.db) y mejores parámetros (.json).
# Al iniciar un runtime nuevo se restauran desde Drive los archivos que falten en local, de modo
# que un estudio interrumpido se reanude; durante la ejecución se copian de vuelta tras cada trial.
import shutil                                               # Copia de archivos entre el disco local y Drive.
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis_Results/Parametrizacion")   # <-- ajusta a tu carpeta en Drive
PATRONES_RESPALDO = ("seeds_parametrizacion.json", "mejores_parametros.json", "*.db")

def sincronizar_drive(restaurar: bool = False) -> None:
    """Copia local -> Drive (o Drive -> local si `restaurar`, sin pisar archivos locales existentes). No hace nada fuera de Colab."""
    if not IN_COLAB:
        return
    try:
        origen, destino = (DRIVE_RESULTS_DIR, RESULTS_DIR) if restaurar else (RESULTS_DIR, DRIVE_RESULTS_DIR)
        destino.mkdir(parents=True, exist_ok=True)
        for patron in PATRONES_RESPALDO:
            for archivo in origen.glob(patron):
                if restaurar and (destino / archivo.name).exists():
                    continue
                shutil.copy2(archivo, destino / archivo.name)
    except Exception as e:                                  # Un fallo de Drive no debe interrumpir la parametrización.
        print(f"[drive] no se pudo {'restaurar' if restaurar else 'respaldar'}: {e}")

sincronizar_drive(restaurar=True)

# Persistencia de semillas: si ya existe el archivo con las semillas de parametrización
# (estudio previo), se cargan desde él en lugar de las recién generadas, de modo que
# un estudio reanudado conserve exactamente las mismas semillas. Si no existe, se crea.
SEEDS_FILE = RESULTS_DIR / "seeds_parametrizacion.json"
if SEEDS_FILE.exists():
    with open(SEEDS_FILE, "r", encoding="utf-8") as f:
        _guardado = json.load(f)
    SEED        = _guardado["seed"]
    SEEDS_PARAM = _guardado["seeds_param"]
    print(f"[semillas] cargadas desde {SEEDS_FILE.name} (generadas el {_guardado['fecha']})")
else:
    with open(SEEDS_FILE, "w", encoding="utf-8") as f:
        json.dump({"fecha": date.today().isoformat(), "seed": SEED, "seeds_param": SEEDS_PARAM}, f, indent=2)
    print(f"[semillas] archivo creado: {SEEDS_FILE.name}")
print(f"[semillas] SEED = {SEED} | {len(SEEDS_PARAM)} semillas de parametrización: {SEEDS_PARAM[:5]} ...")

def storage_url(nombre_variante: str) -> str:
    """Un archivo .db por variante: estudios independientes y fáciles de respaldar."""
    return f"sqlite:///{RESULTS_DIR / (nombre_variante + '.db')}"

def crear_sampler() -> TPESampler:
    """
    Mismo muestreador y misma semilla para TODAS las variantes, de modo que
    la diferencia entre estudios provenga del algoritmo y no del optimizador.
    """
    return TPESampler(seed=SEED, multivariate=True)    # n_startup_trials: valor por defecto de Optuna.

print(f"[resultados] {RESULTS_DIR}")

[entorno] Local → C:\Users\benja\Desktop\workspace\Thesis
[optuna] versión 5.0.0
[semillas] cargadas desde seeds_parametrizacion.json (generadas el 2026-10-29)
[semillas] SEED = 20261029 | 50 semillas de parametrización: [434541565, 913809057, 2119777137, 2062952399, 1254759233] ...
[resultados] C:\Users\benja\Desktop\workspace\Thesis\Results\Parametrizacion


In [3]:
#######################################################
# MODULO DE IMPORTACIÓN DEL CÓDIGO FUENTE
# Se reutilizan los módulos validados en las iteraciones anteriores
# (Notes_1, Notes_2 y Notes_3) sin duplicar su lógica.
# Requiere que SOURCE_CODE_DIR ya esté en sys.path (módulo de ambiente).
#######################################################

############################
# Importaciones propias
############################
import solution_encoding                                    # Se importa el módulo completo: OBJ_FUNCTION_CALLS es un contador global
                                                            # que debe leerse/reiniciarse como solution_encoding.OBJ_FUNCTION_CALLS.
from solution_encoding import load_distance_matrix, random_medoids_pop
from pareto_sorting import non_dominated_sort               # Frentes de Pareto (para obtener F0).
from performance    import hypervolume_f1, HV_REFERENCE_POINT  # Métrica de desempeño: hipervolumen de F0 respecto a HV_REFERENCE_POINT (mayor es mejor).

# Variante base.
from LS import build_neighborhood_matrix                   # M_V de MOLS: GE, BI o BOTH.
from nsga2 import run_nsga2
# Variantes con operadores de búsqueda local clásicos (Notes_2).
from nsga2_variants import (
    run_nsga2_MOPR,                                         # + Multi-Objective Path-Relinking.
    run_nsga2_PLS,                                          # + Pareto Local Search.
    run_nsga2_MOLS,                                         # + L-MOLS / N-MOLS (según `mode`).
)

# Variantes con operadores de información biológica (Notes_3).
from nsga2_variants_bio import (
    run_nsga2_PNS,                                          # + Pareto-Neighborhood Search.
    run_nsga2_DBS,                                          # + Discrepancy-Based Search (según `mode`).
    DBS_MODES,                                              # ("bio", "no_discrepancy", "expresion").
)

############################
# Verificación de importaciones
############################
VARIANTES = {
    "NSGA2":       run_nsga2,
    "NSGA2_PLS":   run_nsga2_PLS,
    "NSGA2_MOLS":  run_nsga2_MOLS,
    "NSGA2_DBS":   run_nsga2_DBS,
    "NSGA2_PNS":   run_nsga2_PNS,
    "NSGA2_MOPR":  run_nsga2_MOPR,
}
print(f"[importación] {len(VARIANTES)} funciones de variantes disponibles: {', '.join(VARIANTES)}")
print(f"[importación] Modos DBS: {DBS_MODES}")

[backend] CuPy detectado → operaciones vectorizadas en GPU.
[backend] CuPy detectado → operaciones vectorizadas en GPU.
[importación] 6 funciones de variantes disponibles: NSGA2, NSGA2_PLS, NSGA2_MOLS, NSGA2_DBS, NSGA2_PNS, NSGA2_MOPR
[importación] Modos DBS: ('bio', 'no_discrepancy', 'expresion')


## Optuna

<div style="text-align: justify;">

Optuna (Akiba et al., 2019) es un _framework_ de código abierto para la optimización automática de hiperparámetros. Su diseño _define-by-run_ permite declarar el espacio de búsqueda dentro de la propia función objetivo, y organiza el trabajo en **estudios** (_study_), compuestos por **ejecuciones** (_trial_) en las que un **muestreador** (_sampler_) propone una configuración que luego se evalúa. Además, almacena el historial de cada estudio en una base de datos y entrega herramientas de visualización para analizarlo. Estas propiedades lo hacen adecuado para este trabajo, donde la evaluación de una configuración es costosa (una corrida completa del _NSGA-II_ por dataset) y se requiere que el proceso sea reproducible y reanudable.

**Metodología.** En este repositorio la parametrización se organiza de la siguiente forma:

1. **Un estudio por variante.** Se crea un estudio independiente para cada una de las seis variantes (`NSGA2`, `NSGA2_PLS`, `NSGA2_MOLS`, `NSGA2_PNS`, `NSGA2_DBS` y `NSGA2_MOPR`), cada uno con su propio espacio de búsqueda, correspondiente a los parámetros marcados como _Requerida_ en la Tabla X. Los parámetros categóricos (por ejemplo, `mode` y `M_v` de MOLS, o `mode` de DBS) se tratan como tales dentro del mismo estudio.
2. **Función objetivo.** En cada _trial_ se muestrea **una única configuración**, que se evalúa en los dos datasets de parametrización (A1 y A2), reiniciando el contador de llamadas a la función objetivo entre ellos para que el presupuesto no se comparta. El valor del _trial_ es el **hipervolumen final del frente $F_0$**, respecto a `HV_REFERENCE_POINT`, promediado entre ambos datasets, y se **maximiza**. Al estar acotado por el área del punto de referencia, es comparable entre datasets de distinto tamaño.
3. **Muestreador.** Todas las variantes usan el mismo muestreador, `TPESampler` (_Tree-structured Parzen Estimator_; Bergstra et al., 2011) en su versión multivariada (Watanabe, 2023), con la misma semilla. El TPE modela por separado la densidad de las configuraciones con buen y mal desempeño y propone nuevas configuraciones donde la razón entre ambas es alta; al ser multivariado, considera las dependencias entre parámetros. Sus primeras ejecuciones (`n_startup_trials`, 10 por defecto en Optuna) se muestrean al azar para iniciar el modelo. De esta forma, las diferencias entre estudios provienen del algoritmo evaluado y no del optimizador.
4. **Semillas compartidas.** Se generan 50 semillas a partir de la fecha del día (AAAAMMDD), que se guardan en `seeds_parametrizacion.json` y son comunes a todas las variantes. El _trial_ $t$ de cualquier variante usa la semilla $t \bmod 50$, por lo que **el _trial_ $t$ de cada variante se evalúa con la misma semilla**, y ambos datasets comparten la semilla del _trial_.
5. **Persistencia y reanudación.** Cada estudio se almacena en un archivo SQLite (`Results/Parametrizacion/<variante>.db`). Si el _runtime_ se interrumpe, al volver a ejecutar solo se completan los _trials_ pendientes. En Google Colab, los estudios, las semillas y los mejores parámetros se respaldan en Drive tras cada _trial_.
6. **Resultado.** Al finalizar, la configuración con mayor hipervolumen de cada estudio se guarda en `mejores_parametros.json` y es la que se utiliza en la etapa de ejecución de los experimentos. Además, se generan los gráficos de historia de optimización, importancia de parámetros y coordenadas paralelas para revisar el comportamiento de cada estudio.

**Justificación de 50 _trials_ por variante.** El número de ejecuciones no se fija por la cantidad de combinaciones posibles, ya que el espacio es mixto y prácticamente continuo, sino mediante el argumento probabilístico de Bergstra y Bengio (2012), que depende de un cuantil de desempeño y no del tamaño del espacio. Con muestreo aleatorio independiente, la probabilidad de que **al menos una** de $n$ configuraciones caiga en el mejor $q$ del espacio es

$$
P = 1 - (1 - q)^n \qquad \Longleftrightarrow \qquad n \ge \frac{\ln(1 - P)}{\ln(1 - q)}
$$

La Tabla Y muestra el número de _trials_ requerido para distintos valores de $q$ y la confianza que entregan 50 _trials_.

**Tabla Y:** _Trials_ necesarios para alcanzar una probabilidad $P$ de que al menos uno caiga en el mejor $q$ del espacio de configuraciones, y probabilidad alcanzada con $n = 50$.

| Mejor fracción ($q$) | $n$ para $P = 95\,\%$ | $n$ para $P = 99\,\%$ | $P$ con $n = 50$ |
|:---:|:---:|:---:|:---:|
| 10 % | 29 | 44 | 99,5 % |
| 5 % | 59 | 90 | 92,3 % |
| 2 % | 149 | 228 | 63,6 % |
| 1 % | 299 | 459 | 39,5 % |

Con $n = 50$ se obtiene una probabilidad de 99,5 % de contar con al menos una configuración dentro del 10 % mejor del espacio, y de 92,3 % dentro del 5 % mejor. Alcanzar el 1 % o el 2 % mejor con alta confianza exigiría 150 a 460 _trials_ por variante, lo que resulta inviable: cada _trial_ equivale a dos corridas completas del _NSGA-II_, y son seis variantes. Por ello, 50 _trials_ representa un compromiso entre la calidad de la configuración encontrada y el costo computacional. Además, coincide con el número de semillas disponibles, de modo que cada semilla se usa exactamente una vez por estudio.

Este argumento debe leerse con tres precauciones. Primero, es una cota para muestreo aleatorio independiente: el TPE no muestrea de esa forma, pero se espera que no rinda peor que la búsqueda aleatoria, por lo que la cota se usa como referencia conservadora y no como garantía exacta. Segundo, el cuantil $q$ se define sobre el desempeño observado, que es ruidoso, ya que cada _trial_ usa una sola semilla; por esta razón, la configuración seleccionada se valida después con las ejecuciones repetidas de la etapa de experimentos. Tercero, la cota no acota la distancia al óptimo global, sino que asegura estar entre las mejores configuraciones del espacio.

**Referencias.**

- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). _Optuna: A next-generation hyperparameter optimization framework_. Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining.
- Bergstra, J., Bardenet, R., Bengio, Y., & Kégl, B. (2011). _Algorithms for hyper-parameter optimization_. Advances in Neural Information Processing Systems, 24.
- Bergstra, J., & Bengio, Y. (2012). _Random search for hyper-parameter optimization_. Journal of Machine Learning Research, 13, 281-305.
- Watanabe, S. (2023). _Tree-structured Parzen estimator: Understanding its algorithm components and their roles for better empirical performance_. arXiv:2304.11127.

</div>


In [4]:
#######################################################
# MODULO DE PARAMETRIZACIÓN CON OPTUNA
# Un estudio por variante. En cada ejecución (trial) se evalúa la MISMA
# configuración sobre los datasets A1 y A2 (presupuesto de llamadas a la función
# objetivo por dataset, sin compartir presupuesto) y se promedia el hipervolumen
# FINAL del frente F0 (respecto a HV_REFERENCE_POINT) (una semilla de SEEDS_PARAM por trial).
#######################################################

############################
# Importaciones
############################
import io                                                   # Silenciar el log por generación de los algoritmos.
import json                                                 # Guardado de la mejor configuración.
import time                                                 # Tiempo por ejecución.
import contextlib                                           # redirect_stdout.
import optuna.visualization as vis                          # plot_optimization_history, plot_param_importances, plot_parallel_coordinate, etc.

############################
# Configuración del usuario
############################
# Número TOTAL de ejecuciones (trials) por metaheurística. Cada una recorre
# ambos datasets. Si el estudio ya existe en disco, solo se completan las que faltan.
N_TRIALS = {
    "NSGA2":        50,
    "NSGA2_PLS":    50,
    "NSGA2_MOLS":   50,
    "NSGA2_DBS":    50,
    "NSGA2_PNS":    50,
    "NSGA2_MOPR":   50,
}

############################
# Constantes del experimento (según la tabla de parametrización)
############################
DATASETS                 = ["A1", "A2"]                  # Datasets de parametrización (prefijo A).
K                        = 4                             # Medoides por solución.
MAX_OBJ_FUNCTION_CALLS   = 100_000                       # Presupuesto POR ejecución y POR dataset.
PERCENTILES_NEIGHBORHOOD = range(10, 36)                 # Percentil de M_V para `neighborhood`: [10, 35].
MV_OPCIONES              = ("GE", "BI", "BOTH")          # Matrices que pueden definir M_V (vecindario de MOLS): expresión, información biológica o sqrt(GE² + BI²).
MOLS_MODES               = ("large", "narrow")           # Criterio de aceptación de MOLS: L-MOLS / N-MOLS.

FUNCIONES_OPTUNA = {
    "NSGA2":        run_nsga2,
    "NSGA2_PLS":    run_nsga2_PLS,
    "NSGA2_MOLS":   run_nsga2_MOLS,
    "NSGA2_DBS":    run_nsga2_DBS,
    "NSGA2_PNS":    run_nsga2_PNS,
    "NSGA2_MOPR":   run_nsga2_MOPR,
}
assert set(N_TRIALS) == set(FUNCIONES_OPTUNA), "N_TRIALS debe definir todas las variantes."

############################
# Carga de datasets (una sola vez)
############################
def cargar_dataset(nombre: str) -> dict:
    """Carga GE/BI y precalcula, para cada opción de M_V, los radios `neighborhood` (percentiles de M_V, sin diagonal)."""
    ge, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DE.npz")
    bi, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DB_M.npz")
    n = ge.shape[0]

    m_v, radios = {}, {}
    for fuente in MV_OPCIONES:
        m_v[fuente] = build_neighborhood_matrix(ge, bi, fuente)
        valores = m_v[fuente][~np.eye(n, dtype=bool)]       # M_V es simétrica: los duplicados no alteran el percentil.
        radios[fuente] = dict(zip(
            PERCENTILES_NEIGHBORHOOD,
            np.percentile(valores, list(PERCENTILES_NEIGHBORHOOD)).astype(float),
        ))
        del valores

    return {"ge": ge, "bi": bi, "m_v": m_v, "n": n, "neighborhood": radios}

DATOS = {nombre: cargar_dataset(nombre) for nombre in DATASETS}

############################
# Función objetivo
############################
def _sugerir_configuracion(trial: optuna.Trial, variante: str) -> dict:
    """Muestrea los hiperparámetros de la variante (rangos de la tabla). Es UNA configuración para ambos datasets."""
    cfg = {
        "pop_size":       trial.suggest_int("pop_size", 50, 300, step=1),
        "crossover_prob": round(trial.suggest_float("crossover_prob", 0.80, 0.90, step=0.01), 2),
        "mutation_prob":  round(trial.suggest_float("mutation_prob",  0.01, 0.05, step=0.01), 2),
    }
    if variante == "NSGA2_PLS":
        cfg["pls_neighborhood"]     = round(trial.suggest_float("pls_neighborhood", 0.05, 0.25, step=0.01), 2)
        cfg["pls_local_budget_pct"] = trial.suggest_int("pls_local_budget_pct", 5, 10)
    elif variante == "NSGA2_MOLS":
        cfg["M_v"]                   = trial.suggest_categorical("M_v", list(MV_OPCIONES))
        cfg["mode"]                  = trial.suggest_categorical("mode", list(MOLS_MODES))
        cfg["mols_local_budget_pct"] = trial.suggest_int("mols_local_budget_pct", 5, 10)
        cfg["neighborhood_pct"]      = trial.suggest_int("neighborhood_pct", 10, 35)
    elif variante == "NSGA2_PNS":
        cfg["nF"]                   = trial.suggest_int("nF", 1, 5)
    elif variante == "NSGA2_DBS":
        cfg["mode"]                 = trial.suggest_categorical("mode", list(DBS_MODES))
        cfg["p_pct"]                = trial.suggest_int("p_pct", 5, 25)
    return cfg

def _kwargs_variante(variante: str, cfg: dict, ds: dict, seed: int) -> dict:
    """Traduce la configuración muestreada a los argumentos reales de la función de la variante."""
    kw = dict(
        ge_matrix=ds["ge"], bi_matrix=ds["bi"], n=ds["n"], k=K,
        pop_size=cfg["pop_size"], max_obj_function_calls=MAX_OBJ_FUNCTION_CALLS,
        crossover_prob=cfg["crossover_prob"], mutation_prob=cfg["mutation_prob"],
        seed=seed,
    )
    budget = lambda pct: MAX_OBJ_FUNCTION_CALLS * pct // 100

    if variante == "NSGA2_PLS":
        kw["pls_neighborhood"] = cfg["pls_neighborhood"]
        kw["pls_local_budget"] = budget(cfg["pls_local_budget_pct"])
    elif variante == "NSGA2_MOLS":
        kw["mode"]              = cfg["mode"]
        kw["m_v_matrix"]        = ds["m_v"][cfg["M_v"]]
        kw["neighborhood"]      = ds["neighborhood"][cfg["M_v"]][cfg["neighborhood_pct"]]
        kw["mols_local_budget"] = budget(cfg["mols_local_budget_pct"])
    elif variante == "NSGA2_PNS":
        kw["nF"]               = cfg["nF"]
    elif variante == "NSGA2_DBS":
        kw["mode"]             = cfg["mode"]
        kw["p"]                = cfg["p_pct"] / 100                 # Percentil p -> fracción.
    return kw

def objective(trial: optuna.Trial, variante: str) -> float:
    """
    Retorna el HIPERVOLUMEN FINAL de F0 (respecto a HV_REFERENCE_POINT), promediado
    entre datasets (se MAXIMIZA). Al estar acotado por el área del punto de referencia,
    es comparable entre A1 y A2. Cada trial usa UNA semilla de SEEDS_PARAM, según su
    número (trial.number): como la lista es la misma para todas las variantes, el
    trial N de cada una se evalúa con la misma semilla. Ambos datasets comparten la
    semilla del trial (2 ejecuciones por trial).
    """
    cfg    = _sugerir_configuracion(trial, variante)
    semilla = SEEDS_PARAM[trial.number % len(SEEDS_PARAM)]
    trial.set_user_attr("semilla", semilla)
    hvs = []

    for nombre, ds in DATOS.items():
        solution_encoding.OBJ_FUNCTION_CALLS = 0            # Reinicio obligatorio: el presupuesto NO se comparte entre datasets.
        t0 = time.time()
        with contextlib.redirect_stdout(io.StringIO()):     # Se silencia el print por generación.
            resultado = FUNCIONES_OPTUNA[variante](**_kwargs_variante(variante, cfg, ds, semilla))

        hv = resultado["history_hv_f1"]
        hvs.append(hv[-1])

        trial.set_user_attr(f"{nombre}_hv_f1_inicial", float(hv[0]))
        trial.set_user_attr(f"{nombre}_hv_f1_final", float(hv[-1]))
        trial.set_user_attr(f"{nombre}_generaciones", int(resultado["generations_run"]))
        trial.set_user_attr(f"{nombre}_llamadas", int(solution_encoding.OBJ_FUNCTION_CALLS))
        trial.set_user_attr(f"{nombre}_segundos", round(time.time() - t0, 1))

    return float(np.mean(hvs))

############################
# Visualización por metaheurística
############################
def mostrar_graficos(estudio: optuna.Study) -> None:
    graficos = {
        "optimization_history": vis.plot_optimization_history,
        "terminator_improvement": vis.plot_terminator_improvement,
        "param_importances":    vis.plot_param_importances,
        "parallel_coordinate":  vis.plot_parallel_coordinate,
    }
    for nombre, funcion in graficos.items():
        try:
            fig = funcion(estudio)
            fig.update_layout(title=f"{estudio.study_name} — {nombre}")
            fig.write_html(RESULTS_DIR / f"{estudio.study_name}_{nombre}.html")
            fig.show()
        except Exception as e:                              # Ej.: muy pocos trials o parámetros sin variación.
            print(f"[{estudio.study_name}] no se pudo generar '{nombre}': {e}")

############################
# Ejecución
############################
def _log_trial(estudio: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    segundos = sum(v for k, v in trial.user_attrs.items() if k.endswith("_segundos"))
    print(f"  [{estudio.study_name}] trial {trial.number:>3} → HV F0 = {trial.value:.4f}  "
          f"({segundos / 60:.1f} min)  {trial.params}")
    sincronizar_drive()                                     # Respaldo en Drive tras cada trial (solo Colab).

MEJORES = {}
for variante in FUNCIONES_OPTUNA:
    estudio = optuna.create_study(
        study_name=variante,
        storage=storage_url(variante),
        sampler=crear_sampler(),
        direction="maximize",                               # Se maximiza el hipervolumen final de F0.
        load_if_exists=True,                                # Permite reanudar un estudio interrumpido.
    )

    completados = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
    pendientes  = max(0, N_TRIALS[variante] - completados)
    print(f"\n════════ {variante}: {completados} completados, {pendientes} pendientes (total {N_TRIALS[variante]}) ════════")

    if pendientes:
        estudio.optimize(lambda t, v=variante: objective(t, v), n_trials=pendientes, callbacks=[_log_trial])

    MEJORES[variante] = {"hv_f1": estudio.best_value, "parametros": estudio.best_params}
    print(f"[{variante}] mejor HV F0 = {estudio.best_value:.4f} | {estudio.best_params}")

    mostrar_graficos(estudio)

with open(RESULTS_DIR / "mejores_parametros.json", "w", encoding="utf-8") as f:
    json.dump(MEJORES, f, indent=2, ensure_ascii=False)
print(f"\n[guardado] {RESULTS_DIR / 'mejores_parametros.json'}")
sincronizar_drive()
if IN_COLAB:
    print(f"[drive] respaldo en {DRIVE_RESULTS_DIR}")

[load] 'A1_S_DE.npz' → matriz 8018×8018. Primer gen: 'A1BG', último: 'ENSG00000285994'.
[load] 'A1_S_DB_M.npz' → matriz 8018×8018. Primer gen: 'A1BG', último: 'ENSG00000285994'.
[load] 'A2_S_DE.npz' → matriz 2705×2705. Primer gen: 'AKT3', último: 'ENSG00000285928'.
[load] 'A2_S_DB_M.npz' → matriz 2705×2705. Primer gen: 'AKT3', último: 'ENSG00000285928'.

════════ NSGA2: 50 completados, 0 pendientes (total 50) ════════
[NSGA2] mejor HV F0 = 90.2587 | {'pop_size': 174, 'crossover_prob': 0.9, 'mutation_prob': 0.04}


C:\Users\benja\AppData\Local\Temp\ipykernel_21788\919914192.py:168: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

e:\Program Files\Python\Lib\site-packages\optuna\visualization\_terminator_improvement.py:97: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

e:\Program Files\Python\Lib\site-packages\optuna\visualization\_terminator_improvement.py:102: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

100%|██████████| 50/50 [00:08<00:00,  6.12it/s]



════════ NSGA2_PLS: 50 completados, 0 pendientes (total 50) ════════
[NSGA2_PLS] mejor HV F0 = 91.0818 | {'pop_size': 217, 'crossover_prob': 0.89, 'mutation_prob': 0.01, 'pls_neighborhood': 0.07, 'pls_local_budget_pct': 9}


C:\Users\benja\AppData\Local\Temp\ipykernel_21788\919914192.py:168: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

e:\Program Files\Python\Lib\site-packages\optuna\visualization\_terminator_improvement.py:97: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

e:\Program Files\Python\Lib\site-packages\optuna\visualization\_terminator_improvement.py:102: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

100%|██████████| 50/50 [00:01<00:00, 35.41it/s]



════════ NSGA2_MOLS: 50 completados, 0 pendientes (total 50) ════════
[NSGA2_MOLS] mejor HV F0 = 90.0614 | {'pop_size': 63, 'crossover_prob': 0.88, 'mutation_prob': 0.05, 'M_v': 'BOTH', 'mode': 'large', 'mols_local_budget_pct': 7, 'neighborhood_pct': 27}


100%|██████████| 50/50 [00:01<00:00, 29.56it/s]


... (3 outputs left out)

100%|██████████| 50/50 [00:01<00:00, 32.39it/s]



════════ NSGA2_MOPR: 50 completados, 0 pendientes (total 50) ════════
[NSGA2_MOPR] mejor HV F0 = 90.6251 | {'pop_size': 292, 'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.05}


100%|██████████| 52/52 [00:01<00:00, 42.64it/s]



[guardado] C:\Users\benja\Desktop\workspace\Thesis\Results\Parametrizacion\mejores_parametros.json
